# ADAM vs GDACS Exposure Comparison

Compares population exposure estimates from ADAM and GDACS to identify systematic differences and understand how agreement varies by country, storm characteristics, and exposure magnitude.

**Comparable thresholds:**
- High wind: ADAM 120 km/h (~65 kt) vs GDACS 64 kt
- Low wind: ADAM 60 km/h (~32 kt) vs GDACS 34 kt

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import wilcoxon, spearmanr, pearsonr
import ocha_stratus as stratus
from dotenv import load_dotenv

load_dotenv()

# Set plot style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

## 1. Load Combined Data

In [ ]:
# Load the combined dataset from the join notebook
df = stratus.load_csv_from_blob('ds-cyclone-exposure/combined_historical_national_exposure.csv')

print(f"Total records: {len(df)}")
print(f"Unique storms: {df['sid'].nunique()}")
print(f"Unique countries: {df['iso3'].nunique()}")

df.head()

## 2. Filter to Cases with Both ADAM and GDACS Data

In [ ]:
# High wind comparison (120 km/h vs 64 kt)
df_high = df[(df['pop_120kmh'].notna()) & (df['pop_64kt'].notna())].copy()
df_high['ratio_high'] = df_high['pop_120kmh'] / df_high['pop_64kt']
df_high['diff_high'] = df_high['pop_120kmh'] - df_high['pop_64kt']
df_high['mean_high'] = (df_high['pop_120kmh'] + df_high['pop_64kt']) / 2
df_high['pct_diff_high'] = 100 * df_high['diff_high'] / df_high['mean_high']

# Low wind comparison (60 km/h vs 34 kt)
df_low = df[(df['pop_60kmh'].notna()) & (df['pop_34kt'].notna())].copy()
df_low['ratio_low'] = df_low['pop_60kmh'] / df_low['pop_34kt']
df_low['diff_low'] = df_low['pop_60kmh'] - df_low['pop_34kt']
df_low['mean_low'] = (df_low['pop_60kmh'] + df_low['pop_34kt']) / 2
df_low['pct_diff_low'] = 100 * df_low['diff_low'] / df_low['mean_low']

print(f"\nHigh wind comparisons (120 km/h vs 64 kt): {len(df_high)}")
print(f"Low wind comparisons (60 km/h vs 34 kt): {len(df_low)}")

# Remove cases where both are zero (undefined ratio)
df_high = df_high[df_high['pop_64kt'] > 0]
df_low = df_low[df_low['pop_34kt'] > 0]

print(f"\nAfter removing zero denominators:")
print(f"High wind comparisons: {len(df_high)}")
print(f"Low wind comparisons: {len(df_low)}")

## 3. Summary Statistics

In [ ]:
def summary_stats(df_comp, adam_col, gdacs_col, ratio_col):
    """Calculate summary statistics for a comparison."""
    stats = {}
    
    # Correlation
    corr_spearman, p_spearman = spearmanr(df_comp[adam_col], df_comp[gdacs_col])
    corr_pearson, p_pearson = pearsonr(df_comp[adam_col], df_comp[gdacs_col])
    
    # Systematic bias test
    diff = df_comp[adam_col] - df_comp[gdacs_col]
    if len(diff) > 0:
        stat_wilcoxon, p_wilcoxon = wilcoxon(diff)
    else:
        stat_wilcoxon, p_wilcoxon = np.nan, np.nan
    
    # Ratio statistics
    ratio = df_comp[ratio_col]
    
    return pd.Series({
        'N': len(df_comp),
        'Spearman r': corr_spearman,
        'Spearman p': p_spearman,
        'Pearson r': corr_pearson,
        'Pearson p': p_pearson,
        'Wilcoxon p': p_wilcoxon,
        'Median Ratio (ADAM/GDACS)': ratio.median(),
        'Mean Ratio': ratio.mean(),
        'Ratio 25th %ile': ratio.quantile(0.25),
        'Ratio 75th %ile': ratio.quantile(0.75),
        '% ADAM > GDACS': 100 * (ratio > 1).mean(),
    })

print("HIGH WIND (120 km/h vs 64 kt):")
print(summary_stats(df_high, 'pop_120kmh', 'pop_64kt', 'ratio_high'))

print("\n" + "="*60 + "\n")

print("LOW WIND (60 km/h vs 34 kt):")
print(summary_stats(df_low, 'pop_60kmh', 'pop_34kt', 'ratio_low'))

## 4. Scatter Plots: ADAM vs GDACS

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# High wind
ax = axes[0]
ax.scatter(df_high['pop_64kt'], df_high['pop_120kmh'], alpha=0.5, s=50)
max_val = max(df_high['pop_64kt'].max(), df_high['pop_120kmh'].max())
ax.plot([0, max_val], [0, max_val], 'r--', label='1:1 line', linewidth=2)
ax.set_xlabel('GDACS 64 kt', fontsize=12)
ax.set_ylabel('ADAM 120 km/h (~65 kt)', fontsize=12)
ax.set_title('High Wind Exposure Comparison', fontsize=14, fontweight='bold')
ax.legend()
ax.set_xscale('log')
ax.set_yscale('log')
ax.grid(True, alpha=0.3)

# Low wind
ax = axes[1]
ax.scatter(df_low['pop_34kt'], df_low['pop_60kmh'], alpha=0.5, s=50, color='orange')
max_val = max(df_low['pop_34kt'].max(), df_low['pop_60kmh'].max())
ax.plot([0, max_val], [0, max_val], 'r--', label='1:1 line', linewidth=2)
ax.set_xlabel('GDACS 34 kt', fontsize=12)
ax.set_ylabel('ADAM 60 km/h (~32 kt)', fontsize=12)
ax.set_title('Low Wind Exposure Comparison', fontsize=14, fontweight='bold')
ax.legend()
ax.set_xscale('log')
ax.set_yscale('log')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Bland-Altman Plots: Bias vs Magnitude

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# High wind
ax = axes[0]
ax.scatter(df_high['mean_high'], df_high['diff_high'], alpha=0.5, s=50)
ax.axhline(0, color='red', linestyle='--', linewidth=2, label='No difference')
ax.axhline(df_high['diff_high'].median(), color='green', linestyle='--', linewidth=2, label=f"Median: {df_high['diff_high'].median():.0f}")
ax.set_xlabel('Average Exposure (ADAM + GDACS) / 2', fontsize=12)
ax.set_ylabel('ADAM - GDACS', fontsize=12)
ax.set_title('High Wind: Bland-Altman Plot', fontsize=14, fontweight='bold')
ax.set_xscale('log')
ax.legend()
ax.grid(True, alpha=0.3)

# Low wind
ax = axes[1]
ax.scatter(df_low['mean_low'], df_low['diff_low'], alpha=0.5, s=50, color='orange')
ax.axhline(0, color='red', linestyle='--', linewidth=2, label='No difference')
ax.axhline(df_low['diff_low'].median(), color='green', linestyle='--', linewidth=2, label=f"Median: {df_low['diff_low'].median():.0f}")
ax.set_xlabel('Average Exposure (ADAM + GDACS) / 2', fontsize=12)
ax.set_ylabel('ADAM - GDACS', fontsize=12)
ax.set_title('Low Wind: Bland-Altman Plot', fontsize=14, fontweight='bold')
ax.set_xscale('log')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Ratio Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# High wind
ax = axes[0]
# Filter out zeros and infinite values
df_high_valid = df_high[df_high['ratio_high'] > 0]
log_ratio_high = np.log10(df_high_valid['ratio_high'])
log_ratio_high = log_ratio_high[np.isfinite(log_ratio_high)]
ax.hist(log_ratio_high, bins=50, alpha=0.7, edgecolor='black')
ax.axvline(0, color='red', linestyle='--', linewidth=2, label='Equal (ratio=1)')
ax.axvline(log_ratio_high.median(), color='green', linestyle='--', linewidth=2, 
           label=f"Median: {10**log_ratio_high.median():.2f}")
ax.set_xlabel('log10(ADAM / GDACS)', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.set_title('High Wind: Distribution of Ratios', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Low wind
ax = axes[1]
# Filter out zeros and infinite values
df_low_valid = df_low[df_low['ratio_low'] > 0]
log_ratio_low = np.log10(df_low_valid['ratio_low'])
log_ratio_low = log_ratio_low[np.isfinite(log_ratio_low)]
ax.hist(log_ratio_low, bins=50, alpha=0.7, color='orange', edgecolor='black')
ax.axvline(0, color='red', linestyle='--', linewidth=2, label='Equal (ratio=1)')
ax.axvline(log_ratio_low.median(), color='green', linestyle='--', linewidth=2,
           label=f"Median: {10**log_ratio_low.median():.2f}")
ax.set_xlabel('log10(ADAM / GDACS)', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.set_title('Low Wind: Distribution of Ratios', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Comparison by Country

In [ ]:
# Countries with at least 3 comparisons
country_counts = df_high['country_name'].value_counts()
countries_to_plot = country_counts[country_counts >= 3].index[:15]  # Top 15

df_high_countries = df_high[df_high['country_name'].isin(countries_to_plot)]

fig, ax = plt.subplots(figsize=(12, 8))
df_high_countries.boxplot(column='ratio_high', by='country_name', ax=ax, rot=45)
ax.axhline(1, color='red', linestyle='--', linewidth=2, label='Equal (ratio=1)')
ax.set_xlabel('Country', fontsize=12)
ax.set_ylabel('ADAM / GDACS Ratio (High Wind)', fontsize=12)
ax.set_title('High Wind Exposure Ratios by Country (≥3 comparisons)', fontsize=14, fontweight='bold')
ax.legend()
plt.suptitle('')  # Remove default title
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Summary by country
country_summary = df_high_countries.groupby('country_name').agg({
    'ratio_high': ['count', 'median', 'mean'],
    'pct_diff_high': 'median'
}).round(2)
country_summary.columns = ['N', 'Median Ratio', 'Mean Ratio', 'Median % Diff']
country_summary = country_summary.sort_values('Median Ratio', ascending=False)

print("\nCountry-level summary (High Wind):")
print(country_summary)

## 8. Comparison by Alert Level (Storm Intensity)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Check if alert_level column exists
if 'alert_level' in df_high.columns:
    # High wind
    ax = axes[0]
    alert_order = ['Red', 'Orange', 'Green']
    df_high_alert = df_high[df_high['alert_level'].isin(alert_order)]
    sns.boxplot(data=df_high_alert, x='alert_level', y='ratio_high', order=alert_order, ax=ax)
    ax.axhline(1, color='red', linestyle='--', linewidth=2, label='Equal (ratio=1)')
    ax.set_xlabel('Alert Level', fontsize=12)
    ax.set_ylabel('ADAM / GDACS Ratio', fontsize=12)
    ax.set_title('High Wind: Ratios by Alert Level', fontsize=14, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Low wind
    ax = axes[1]
    df_low_alert = df_low[df_low['alert_level'].isin(alert_order)]
    sns.boxplot(data=df_low_alert, x='alert_level', y='ratio_low', order=alert_order, ax=ax)
    ax.axhline(1, color='red', linestyle='--', linewidth=2, label='Equal (ratio=1)')
    ax.set_xlabel('Alert Level', fontsize=12)
    ax.set_ylabel('ADAM / GDACS Ratio', fontsize=12)
    ax.set_title('Low Wind: Ratios by Alert Level', fontsize=14, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Summary by alert level
    print("\nAlert level summary (High Wind):")
    print(df_high_alert.groupby('alert_level')['ratio_high'].agg(['count', 'median', 'mean']).round(2))
else:
    print("Note: 'alert_level' column not found in dataset - skipping alert level analysis")
    print("Available columns:", df_high.columns.tolist())

## 9. Ratio vs Storm Size (Total Exposure)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# High wind
ax = axes[0]
df_high['total_exposure'] = df_high['pop_120kmh'] + df_high['pop_64kt']
ax.scatter(df_high['total_exposure'], df_high['ratio_high'], alpha=0.5, s=50)
ax.axhline(1, color='red', linestyle='--', linewidth=2, label='Equal (ratio=1)')
ax.set_xlabel('Total Exposure (ADAM + GDACS)', fontsize=12)
ax.set_ylabel('ADAM / GDACS Ratio', fontsize=12)
ax.set_title('High Wind: Ratio vs Storm Size', fontsize=14, fontweight='bold')
ax.set_xscale('log')
ax.legend()
ax.grid(True, alpha=0.3)

# Low wind
ax = axes[1]
df_low['total_exposure'] = df_low['pop_60kmh'] + df_low['pop_34kt']
ax.scatter(df_low['total_exposure'], df_low['ratio_low'], alpha=0.5, s=50, color='orange')
ax.axhline(1, color='red', linestyle='--', linewidth=2, label='Equal (ratio=1)')
ax.set_xlabel('Total Exposure (ADAM + GDACS)', fontsize=12)
ax.set_ylabel('ADAM / GDACS Ratio', fontsize=12)
ax.set_title('Low Wind: Ratio vs Storm Size', fontsize=14, fontweight='bold')
ax.set_xscale('log')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 10. Temporal Trends

In [ ]:
# Extract year from from_date
df_high['year'] = pd.to_datetime(df_high['from_date']).dt.year
df_low['year'] = pd.to_datetime(df_low['from_date']).dt.year

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# High wind
ax = axes[0]
yearly_high = df_high.groupby('year')['ratio_high'].median()
ax.plot(yearly_high.index, yearly_high.values, marker='o', linewidth=2, markersize=8)
ax.axhline(1, color='red', linestyle='--', linewidth=2, label='Equal (ratio=1)')
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Median ADAM / GDACS Ratio', fontsize=12)
ax.set_title('High Wind: Temporal Trend in Ratios', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Low wind
ax = axes[1]
yearly_low = df_low.groupby('year')['ratio_low'].median()
ax.plot(yearly_low.index, yearly_low.values, marker='o', linewidth=2, markersize=8, color='orange')
ax.axhline(1, color='red', linestyle='--', linewidth=2, label='Equal (ratio=1)')
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Median ADAM / GDACS Ratio', fontsize=12)
ax.set_title('Low Wind: Temporal Trend in Ratios', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 11. Identify Largest Disagreements

In [ ]:
# Top 10 cases where ADAM >> GDACS
print("Top 10 cases where ADAM substantially exceeds GDACS (High Wind):")
top_adam = df_high.nlargest(10, 'ratio_high')[['storm_name', 'country_name', 'pop_120kmh', 'pop_64kt', 'ratio_high', 'season']]
print(top_adam.to_string(index=False))

print("\n" + "="*80 + "\n")

# Top 10 cases where GDACS >> ADAM
print("Top 10 cases where GDACS substantially exceeds ADAM (High Wind):")
top_gdacs = df_high.nsmallest(10, 'ratio_high')[['storm_name', 'country_name', 'pop_120kmh', 'pop_64kt', 'ratio_high', 'season']]
print(top_gdacs.to_string(index=False))

## 12. Save Analysis Results

In [ ]:
# Save comparison datasets for further analysis
df_high[['sid', 'storm_name', 'country_name', 'season', 'alert_level', 
         'pop_120kmh', 'pop_64kt', 'ratio_high', 'diff_high', 'pct_diff_high']].to_csv(
    'adam_gdacs_comparison_high_wind.csv', index=False
)

df_low[['sid', 'storm_name', 'country_name', 'season', 'alert_level',
        'pop_60kmh', 'pop_34kt', 'ratio_low', 'diff_low', 'pct_diff_low']].to_csv(
    'adam_gdacs_comparison_low_wind.csv', index=False
)

print("Comparison datasets saved!")
print("  - adam_gdacs_comparison_high_wind.csv")
print("  - adam_gdacs_comparison_low_wind.csv")